# 02 - Ingestion : API USGS → data/landing
Télécharge le catalogue mondial des séismes de 2016 à 2025, semaine par semaine, en CSV.
L'API refuse les réponses de plus de 20 000 événements : une semaine en contient environ 3 000.


## 1. Imports et paramètres



In [1]:
import time
import urllib.request
from datetime import date, timedelta
from pathlib import Path

DEBUT = date(2016, 1, 1)
FIN = date(2026, 1, 1)          # non incluse
RACINE = Path("/work/data/landing/seismes")   # = data/landing/seismes dans PC
URL = "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&starttime={debut}&endtime={fin}&orderby=time-asc"
UA = {"User-Agent": "projet-etudiant-seismes/1.0"}
LIMITE = 20000                  # maximum accepté par l'API par requête

print("Dossier de destination :", RACINE)
print("Fichiers déjà présents :", len(list(RACINE.glob("*/*.csv"))))


Dossier de destination : /work/data/landing/seismes
Fichiers déjà présents : 521


## 2. Fonction : télécharger une période dans un fichier

In [2]:
def telecharger(debut, fin, cible):
    """Télécharge les séismes entre debut et fin dans cible.
    Renvoie le nombre de séismes (sans l'en-tête), ou -1 si échec."""
    url = URL.format(debut=debut, fin=fin)
    tmp = cible.with_suffix(".part")
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=120) as r:
            contenu = r.read()
        with open(tmp, "wb") as f:
            f.write(contenu)
    except Exception as e:
        tmp.unlink(missing_ok=True)
        print(f"{cible.name} : échec ({e})")
        return -1
    tmp.replace(cible)
    time.sleep(0.5)  # petite pause pour ne pas surcharger l'API
    return contenu.count(b"\n") - 1


## 3. Fonction : ingérer toutes les semaines entre deux dates


In [3]:
def ingerer(date_debut, date_fin):
    nb_telecharges = 0
    nb_deja_la = 0
    nb_seismes = 0
    echecs = 0

    debut = date_debut
    while debut < date_fin:
        fin = min(debut + timedelta(days=7), date_fin)
        dossier = RACINE / str(debut.year)
        dossier.mkdir(parents=True, exist_ok=True)
        cible = dossier / f"seismes_{debut}.csv"

        if cible.exists() and cible.stat().st_size > 0:
            nb_deja_la += 1
        else:
            nb = telecharger(debut, fin, cible)
            if nb >= LIMITE:
                print(f"ATTENTION {cible.name} : {nb} lignes, découpage par jour")
                cible.unlink()
                jour = debut
                while jour < fin:
                    nb_jour = telecharger(jour, jour + timedelta(days=1), dossier / f"seismes_{jour}_jour.csv")
                    if nb_jour < 0:
                        echecs += 1
                    else:
                        nb_seismes += nb_jour
                    jour = jour + timedelta(days=1)
            elif nb < 0:
                echecs += 1
            else:
                nb_telecharges += 1
                nb_seismes += nb
                print(f"{cible.name} : {nb} séismes téléchargés")
        debut = fin

    print(f"Semaines déjà présentes (sautées) : {nb_deja_la}")
    print(f"Semaines téléchargées             : {nb_telecharges} ({nb_seismes} séismes)")
    print(f"Échecs                            : {echecs}  -> si > 0, relancer la cellule")


## 4. Lancement sur 2016-2025 (reprise après interruption)

In [4]:
t0 = time.time()
ingerer(DEBUT, FIN)
print(f"Durée : {time.time() - t0:.1f} s")


seismes_2025-12-26.csv : 2252 séismes téléchargés
Semaines déjà présentes (sautées) : 521
Semaines téléchargées             : 1 (2252 séismes)
Échecs                            : 0  -> si > 0, relancer la cellule
Durée : 5.8 s


## 5. Vérification : fichiers, événements et volume par année

In [ ]:
total_fichiers = 0
total_evenements = 0
total_octets = 0

print(f"{'année':<6} {'fichiers':>8} {'événements':>11}")
for dossier in sorted(RACINE.iterdir()):
    fichiers = list(dossier.glob("*.csv"))
    evenements = 0
    for f in fichiers:
        with open(f, "rb") as fh:
            evenements += sum(1 for _ in fh) - 1   # -1 pour l'en-tête
        total_octets += f.stat().st_size
    total_fichiers += len(fichiers)
    total_evenements += evenements
    print(f"{dossier.name:<6} {len(fichiers):>8} {evenements:>11}")

print(f"\nTOTAL : {total_fichiers} fichiers, {total_evenements} événements, {total_octets / 1e6:.0f} Mo")
print("Attendu (étape 1 du journal) : 522 fichiers, 1 604 291 événements, ~300 Mo")


année  fichiers  événements
2016         53      128803
2017         52      133182
2018         52      190128
2019         52      189040
2020         52      200844
2021         53      165603
2022         52      150707
2023         52      151013
2024         52      155428
2025         52      139543

TOTAL : 522 fichiers, 1604291 événements, 300 Mo
Attendu (étape 1 du journal) : 522 fichiers, 1 604 291 événements, ~300 Mo
